# Example solution for CNN exercise

First, let us recycle the code from the first notebook in which we used the MNIST dataset

We want to process the images with a CNN this time, and then add a linear net at the end to get our classification

In [11]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 6, 5, padding=2),
            nn.Sigmoid(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(6, 16, 5),
            nn.Sigmoid(),
            nn.MaxPool2d(2, stride=2),
            nn.Flatten(),
            nn.Linear(400,120),
            nn.Sigmoid(),
            nn.Linear(120,84),
            nn.Sigmoid(),
            nn.Linear(84,10),
        )

    def forward(self, x):
        logits = self.network(x)
        return logits

Again recycling from notebook 5

In [16]:
# Network hyperparameters
learning_rate = 0.5
batch_size = 64
epochs = 10
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

In [17]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    print(f"current learning rate: {learning_rate}")
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    lr_scheduler.step()


In [18]:
def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [19]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
current learning rate: 0.5
loss: 2.301043  [   64/60000]
loss: 2.294551  [ 6464/60000]
loss: 2.315758  [12864/60000]
loss: 2.319354  [19264/60000]
loss: 2.315545  [25664/60000]
loss: 2.311156  [32064/60000]
loss: 2.302520  [38464/60000]
loss: 2.306277  [44864/60000]
loss: 2.309309  [51264/60000]
loss: 2.291247  [57664/60000]
Test Error: 
 Accuracy: 10.3%, Avg loss: 2.301771 

Epoch 2
-------------------------------
current learning rate: 0.5
loss: 2.302934  [   64/60000]
loss: 2.296573  [ 6464/60000]
loss: 2.317266  [12864/60000]
loss: 2.309542  [19264/60000]
loss: 2.311936  [25664/60000]
loss: 2.310240  [32064/60000]
loss: 2.300480  [38464/60000]
loss: 2.308590  [44864/60000]
loss: 2.307277  [51264/60000]
loss: 2.293010  [57664/60000]
Test Error: 
 Accuracy: 10.3%, Avg loss: 2.301503 

Epoch 3
-------------------------------
current learning rate: 0.5
loss: 2.302412  [   64/60000]
loss: 2.297124  [ 6464/60000]
loss: 2.315942  [12864/60000]
loss: